In [1]:
import os

# Move up one level to set the working directory to the repo root
os.chdir(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [3]:
!pip install goto-conversion

  Using cached goto_conversion-2.0.2-py3-none-any.whl.metadata (7.1 kB)
Using cached goto_conversion-2.0.2-py3-none-any.whl (6.2 kB)


In [2]:
import pandas as pd
import numpy as np
import re
import goto_conversion

## Create `missing_odds_results.csv`

In [101]:
# Load datasets
betexplorer_df = pd.read_csv(r"ncaam_betexplorer_odds.csv")
tourney_results_df = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
regular_season_results_df = pd.read_csv(r"data\kaggle\MRegularSeasonCompactResults.csv")
seasons_df = pd.read_csv(r"data\kaggle\MSeasons.csv")
teams_df = pd.read_csv(r"data\kaggle\MTeams.csv")
team_spellings_df = pd.read_csv(r"data\TeamSpellings.csv", encoding="ISO-8859-1")

# Drop unnecessary columns from team_spellings_df
team_spellings_df = team_spellings_df.drop(
    columns=["WTeamID", "espn_id"], errors="ignore"
)

# Combine tournament and regular season results
compact_results_df = pd.concat(
    [tourney_results_df, regular_season_results_df], ignore_index=True
)

# Merge with seasons_df to get DayZero and calculate actual date
compact_results_df = compact_results_df.merge(
    seasons_df[["Season", "DayZero"]], on="Season", how="left"
)

# Convert DayZero to datetime and calculate actual matchup date
compact_results_df["DayZero"] = pd.to_datetime(compact_results_df["DayZero"])
compact_results_df["match_date"] = compact_results_df["DayZero"] + pd.to_timedelta(
    compact_results_df["DayNum"], unit="D"
)

# Merge Team Names into compact_results_df
compact_results_df = compact_results_df.merge(
    teams_df[["TeamID", "TeamName"]], left_on="WTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "WTeamName"})
compact_results_df = compact_results_df.merge(
    teams_df[["TeamID", "TeamName"]], left_on="LTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "LTeamName"})

# Drop unnecessary TeamID columns after merging team names
compact_results_df = compact_results_df.drop(
    columns=["TeamID_x", "TeamID_y"], errors="ignore"
)


# Function to parse team1_score and team2_score from BetExplorer Result
def parse_scores(result):
    match = re.search(r"(\d+):(\d+)", str(result))
    if match:
        return int(match.group(1)), int(match.group(2))
    return None, None


# Apply parsing function
betexplorer_df[["team1_score", "team2_score"]] = betexplorer_df["Result"].apply(
    lambda x: pd.Series(parse_scores(x))
)

# Convert the 'Date' column directly to datetime format
betexplorer_df["match_date"] = pd.to_datetime(betexplorer_df["Date"], errors="coerce")

# Convert Team1 and Team2 to lowercase
betexplorer_df["Team1"] = betexplorer_df["Team1"].str.lower()
betexplorer_df["Team2"] = betexplorer_df["Team2"].str.lower()

# Map Team Names from TeamSpellings.csv to MTeamID
team_spellings_df = team_spellings_df.rename(columns={"TeamNameSpelling": "TeamName"})
betexplorer_df = betexplorer_df.merge(
    team_spellings_df, left_on="Team1", right_on="TeamName", how="left"
).rename(columns={"TeamID": "MTeamID_x"})
betexplorer_df = betexplorer_df.merge(
    team_spellings_df, left_on="Team2", right_on="TeamName", how="left"
).rename(columns={"TeamID": "MTeamID_y"})

# Create WTeamID and LTeamID based on scores
betexplorer_df["WTeamID"] = betexplorer_df.apply(
    lambda row: row["MTeamID_x"]
    if row["team1_score"] > row["team2_score"]
    else row["MTeamID_y"],
    axis=1,
)
betexplorer_df["LTeamID"] = betexplorer_df.apply(
    lambda row: row["MTeamID_y"]
    if row["team1_score"] > row["team2_score"]
    else row["MTeamID_x"],
    axis=1,
)

# Create WScore and LScore based on max/min of team1_score and team2_score
betexplorer_df["WScore"] = betexplorer_df[["team1_score", "team2_score"]].max(axis=1)
betexplorer_df["LScore"] = betexplorer_df[["team1_score", "team2_score"]].min(axis=1)

# Drop unnecessary MTeamID columns after determining WTeamID and LTeamID
betexplorer_df = betexplorer_df.drop(
    columns=["MTeamID_x", "MTeamID_y", "team1_score", "team2_score"], errors="ignore"
)

In [102]:
compact_results_df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,match_date,WTeamName,LTeamName
0,1985,136,1116,63,1234,54,N,0,1984-10-29,1985-03-14,Arkansas,Iowa
1,1985,136,1120,59,1345,58,N,0,1984-10-29,1985-03-14,Auburn,Purdue
2,1985,136,1207,68,1250,43,N,0,1984-10-29,1985-03-14,Georgetown,Lehigh
3,1985,136,1229,58,1425,55,N,0,1984-10-29,1985-03-14,Illinois St,USC
4,1985,136,1242,49,1325,38,N,0,1984-10-29,1985-03-14,Kansas,Ohio


In [103]:
betexplorer_df.head()

,Season,Month,Date,Team1,Team2,Result,Odds1,Odds2,match_date,TeamName_x,TeamName_y,WTeamID,LTeamID,WScore,LScore
0,2008-2009,November,2008-11-30,air force,norfolk state,75:52,,,2008-11-30,air force,norfolk state,1102.0,1313.0,75,52
1,2008-2009,November,2008-11-30,arizona,northern arizona,74:57,,,2008-11-30,arizona,northern arizona,1112.0,1319.0,74,57
2,2008-2009,November,2008-11-30,baylor,wake forest,74:87,,,2008-11-30,baylor,wake forest,1448.0,1124.0,87,74
3,2008-2009,November,2008-11-30,central arkansas,south dakota st.,67:72,,,2008-11-30,central arkansas,south dakota st.,1355.0,1146.0,72,67
4,2008-2009,November,2008-11-30,central connecticut state,lafayette,82:78,,,2008-11-30,central connecticut state,lafayette,1148.0,1248.0,82,78


In [104]:
# Merge BetExplorer with compact results on match_date and team IDs
merged_df = compact_results_df.merge(
    betexplorer_df,
    left_on=["match_date", "WTeamID", "LTeamID"],
    right_on=["match_date", "WTeamID", "LTeamID"],
    how="left",
    suffixes=("_compact", "_bet"),
)

In [105]:
merged_df.head()

,Season_compact,DayNum,WTeamID,WScore_compact,LTeamID,LScore_compact,WLoc,NumOT,DayZero,match_date,...,Date,Team1,Team2,Result,Odds1,Odds2,TeamName_x,TeamName_y,WScore_bet,LScore_bet
0,1985,136,1116,63,1234,54,N,0,1984-10-29,1985-03-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1985,136,1120,59,1345,58,N,0,1984-10-29,1985-03-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1985,136,1207,68,1250,43,N,0,1984-10-29,1985-03-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1985,136,1229,58,1425,55,N,0,1984-10-29,1985-03-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1985,136,1242,49,1325,38,N,0,1984-10-29,1985-03-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [107]:
# Identify matchups from compact results (season 2008 and greater) that don't have corresponding BetExplorer odds data
missing_odds_df = merged_df[
    (merged_df["Season_compact"] >= 2009)
    & (merged_df["Season_compact"] < 2025)
    & merged_df["Result"].isna()
]

In [108]:
# Select final columns for display
final_columns = [
    "Season_compact",
    "DayNum",
    "WTeamID",
    "WScore_compact",
    "LTeamID",
    "LScore_compact",
    "WLoc",
    "NumOT",
    "DayZero",
    "match_date",
    "WTeamName",
    "LTeamName",
]
missing_odds_df = missing_odds_df[final_columns]

In [109]:
# Display only a sample of the missing matchups dataset
missing_odds_df.head(1)

,Season_compact,DayNum,WTeamID,WScore_compact,LTeamID,LScore_compact,WLoc,NumOT,DayZero,match_date,WTeamName,LTeamName
1628,2010,139,1326,75,1210,66,N,0,2009-11-02,2010-03-21,Ohio St,Georgia Tech


In [112]:
# Output missing matchups to CSV
missing_odds_df.to_csv("missing_odds_results.csv", index=False)

In [110]:
len(compact_results_df)

194314

In [111]:
len(missing_odds_df)

1246

In [124]:
len(merged_df[merged_df["Result"].notna()])

84168

## Create `still_missing_odds_results.csv` and `manual_check_missing_odds_results.csv` and `score_matched_results.csv`

In [113]:
# Try to find additional matches by merging on match_date and scores
score_matched_df = missing_odds_df.merge(
    betexplorer_df,
    left_on=["match_date", "WScore_compact", "LScore_compact"],
    right_on=["match_date", "WScore", "LScore"],
    how="left",
    suffixes=("_missing", "_bet"),
)

In [114]:
score_matched_df.head()

,Season_compact,DayNum,WTeamID_missing,WScore_compact,LTeamID_missing,LScore_compact,WLoc,NumOT,DayZero,match_date,...,Team2,Result,Odds1,Odds2,TeamName_x,TeamName_y,WTeamID_bet,LTeamID_bet,WScore,LScore
0,2010,139,1326,75,1210,66,N,0,2009-11-02,2010-03-21,...,georgia state,75:66,,,ohio state,georgia state,1326.0,1209.0,75.0,66.0
1,2009,11,1222,73,1443,64,H,0,2008-11-03,2008-11-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2009,11,1362,75,1218,70,A,0,2008-11-03,2008-11-14,...,eastern oregon,75:70,,,idaho state,NaN,1226.0,NaN,75.0,70.0
3,2009,14,1218,67,1226,64,H,1,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,14,1272,80,1269,58,H,0,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [115]:
# Separate still missing matches and those with multiple matches
still_missing_odds_df = score_matched_df[score_matched_df["Result"].isna()]
manual_check_missing_odds_df = score_matched_df[
    score_matched_df.duplicated(subset=["match_date", "WScore", "LScore"], keep=False)
]

In [116]:
# Display only a sample of the missing matchups datasets
still_missing_odds_df.head()

,Season_compact,DayNum,WTeamID_missing,WScore_compact,LTeamID_missing,LScore_compact,WLoc,NumOT,DayZero,match_date,...,Team2,Result,Odds1,Odds2,TeamName_x,TeamName_y,WTeamID_bet,LTeamID_bet,WScore,LScore
1,2009,11,1222,73,1443,64,H,0,2008-11-03,2008-11-14,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2009,14,1218,67,1226,64,H,1,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,14,1272,80,1269,58,H,0,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2009,14,1388,99,1201,85,H,0,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2009,21,1332,92,1104,69,N,0,2008-11-03,2008-11-24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [117]:
manual_check_missing_odds_df.head()

,Season_compact,DayNum,WTeamID_missing,WScore_compact,LTeamID_missing,LScore_compact,WLoc,NumOT,DayZero,match_date,...,Team2,Result,Odds1,Odds2,TeamName_x,TeamName_y,WTeamID_bet,LTeamID_bet,WScore,LScore
3,2009,14,1218,67,1226,64,H,1,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2009,14,1272,80,1269,58,H,0,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2009,14,1388,99,1201,85,H,0,2008-11-03,2008-11-17,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
10,2009,82,1370,99,1188,82,H,0,2008-11-03,2009-01-24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
11,2009,82,1377,98,1315,77,A,0,2008-11-03,2009-01-24,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [123]:
len(
    compact_results_df[
        (compact_results_df["Season"] >= 2009) & (compact_results_df["Season"] < 2025)
    ]
)

85414

In [118]:
len(still_missing_odds_df)

985

In [119]:
len(manual_check_missing_odds_df)

763

In [120]:
# Output missing matchups to CSV
still_missing_odds_df.to_csv("still_missing_odds_results.csv", index=False)
manual_check_missing_odds_df.to_csv(
    "manual_check_missing_odds_results.csv", index=False
)

In [125]:
score_matched_df.to_csv("score_matched_results.csv", index=False)

## Create Compact Results with Odds Datasets

In [5]:
import pandas as pd
import re

# Load datasets
betexplorer_df = pd.read_csv(r"data\betexplorer\ncaam_betexplorer_odds.csv")
tourney_results_df = pd.read_csv(r"data\kaggle\MNCAATourneyCompactResults.csv")
regular_season_results_df = pd.read_csv(r"data\kaggle\MRegularSeasonCompactResults.csv")
seasons_df = pd.read_csv(r"data\kaggle\MSeasons.csv")
teams_df = pd.read_csv(r"data\kaggle\MTeams.csv")
team_spellings_df = pd.read_csv(r"data\TeamSpellings.csv", encoding="ISO-8859-1")
manual_odds_df = pd.read_csv(r"data\betexplorer\manual_odds_data.csv")

# Drop unnecessary columns from team_spellings_df
team_spellings_df = team_spellings_df.drop(
    columns=["WTeamID", "espn_id"], errors="ignore"
)


def process_betexplorer_data(betexplorer_df, team_spellings_df):
    """
    Processes the BetExplorer data by mapping team names to IDs, parsing scores,
    and determining winning/losing teams and odds.
    """
    # Convert Team1 and Team2 to lowercase
    betexplorer_df["Team1"] = betexplorer_df["Team1"].str.lower()
    betexplorer_df["Team2"] = betexplorer_df["Team2"].str.lower()

    # Map Team Names from TeamSpellings.csv to MTeamID
    team_spellings_df = team_spellings_df.rename(
        columns={"TeamNameSpelling": "TeamName"}
    )
    betexplorer_df = betexplorer_df.merge(
        team_spellings_df, left_on="Team1", right_on="TeamName", how="left"
    ).rename(columns={"TeamID": "MTeamID_x"})
    betexplorer_df = betexplorer_df.merge(
        team_spellings_df, left_on="Team2", right_on="TeamName", how="left"
    ).rename(columns={"TeamID": "MTeamID_y"})

    # Function to parse team1_score and team2_score from BetExplorer Result
    def parse_scores(result):
        match = re.search(r"(\d+):(\d+)", str(result))
        if match:
            return int(match.group(1)), int(match.group(2))
        return None, None

    # Apply parsing function
    betexplorer_df[["team1_score", "team2_score"]] = betexplorer_df["Result"].apply(
        lambda x: pd.Series(parse_scores(x))
    )

    # Convert the 'Date' column directly to datetime format
    betexplorer_df["match_date"] = pd.to_datetime(
        betexplorer_df["Date"], errors="coerce"
    )

    # Create WTeamID, LTeamID, WOdds, and LOdds based on scores
    betexplorer_df["WTeamID"] = betexplorer_df.apply(
        lambda row: row["MTeamID_x"]
        if row["team1_score"] > row["team2_score"]
        else row["MTeamID_y"],
        axis=1,
    )
    betexplorer_df["LTeamID"] = betexplorer_df.apply(
        lambda row: row["MTeamID_y"]
        if row["team1_score"] > row["team2_score"]
        else row["MTeamID_x"],
        axis=1,
    )
    betexplorer_df["WOdds"] = betexplorer_df.apply(
        lambda row: row["Odds1"]
        if row["team1_score"] > row["team2_score"]
        else row["Odds2"],
        axis=1,
    )
    betexplorer_df["LOdds"] = betexplorer_df.apply(
        lambda row: row["Odds2"]
        if row["team1_score"] > row["team2_score"]
        else row["Odds1"],
        axis=1,
    )

    # Drop unnecessary columns
    betexplorer_df = betexplorer_df.drop(
        columns=[
            "MTeamID_x",
            "MTeamID_y",
            "TeamName_x",
            "TeamName_y",
            "team1_score",
            "team2_score",
        ],
        errors="ignore",
    )

    return betexplorer_df


# Process BetExplorer data
betexplorer_df = process_betexplorer_data(betexplorer_df, team_spellings_df)


def generate_compact_results_with_odds(compact_results_df, betexplorer_df):
    """
    This function takes a compact results dataframe (i.e. tourney_results_df or regular_season_results_df)
    and betexplorer_df, then outputs a dataframe with the following columns:
    'Season', 'DayNum', 'WTeamID', 'WScore', 'LTeamID', 'LScore', 'WLoc', 'NumOT', 'DayZero', 'WOdds', 'LOdds', 'match_date'
    """
    # Merge with seasons_df to get DayZero
    compact_results_df = compact_results_df.merge(
        seasons_df[["Season", "DayZero"]], on="Season", how="left"
    )

    # Convert DayZero to datetime and calculate actual matchup date
    compact_results_df["DayZero"] = pd.to_datetime(compact_results_df["DayZero"])
    compact_results_df["match_date"] = compact_results_df["DayZero"] + pd.to_timedelta(
        compact_results_df["DayNum"], unit="D"
    )

    # Merge BetExplorer odds data with compact results based on match_date, WTeamID, and LTeamID
    merged_df = compact_results_df.merge(
        betexplorer_df[["WTeamID", "LTeamID", "WOdds", "LOdds", "match_date"]],
        on=["match_date", "WTeamID", "LTeamID"],
        how="left",
    )

    # Select final columns for output
    final_columns = [
        "Season",
        "DayNum",
        "WTeamID",
        "WScore",
        "LTeamID",
        "LScore",
        "WLoc",
        "NumOT",
        "DayZero",
        "WOdds",
        "LOdds",
        "match_date",
    ]
    return merged_df[final_columns]

In [6]:
regular_season_results_odds_df = generate_compact_results_with_odds(
    regular_season_results_df, betexplorer_df
)
tourney_results_odds_df = generate_compact_results_with_odds(
    tourney_results_df, betexplorer_df
)

# Merge manually added odds data
regular_season_results_odds_df = regular_season_results_odds_df.merge(
    manual_odds_df[["Season", "DayNum", "WTeamID", "LTeamID", "WOdds", "LOdds"]],
    on=["Season", "DayNum", "WTeamID", "LTeamID"],
    how="left",
    suffixes=("", "_manual"),
)

# Fill missing odds with manually added odds where available
regular_season_results_odds_df["WOdds"] = regular_season_results_odds_df[
    "WOdds"
].combine_first(regular_season_results_odds_df["WOdds_manual"])
regular_season_results_odds_df["LOdds"] = regular_season_results_odds_df[
    "LOdds"
].combine_first(regular_season_results_odds_df["LOdds_manual"])

# Drop manual columns after merging
regular_season_results_odds_df = regular_season_results_odds_df.drop(
    columns=["WOdds_manual", "LOdds_manual"], errors="ignore"
)

In [7]:
tourney_results_odds_df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date
0,1985,136,1116,63,1234,54,N,0,1984-10-29,NaN,NaN,1985-03-14
1,1985,136,1120,59,1345,58,N,0,1984-10-29,NaN,NaN,1985-03-14
2,1985,136,1207,68,1250,43,N,0,1984-10-29,NaN,NaN,1985-03-14
3,1985,136,1229,58,1425,55,N,0,1984-10-29,NaN,NaN,1985-03-14
4,1985,136,1242,49,1325,38,N,0,1984-10-29,NaN,NaN,1985-03-14


In [8]:
regular_season_results_odds_df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date
0,1985,20,1228,81,1328,64,N,0,1984-10-29,NaN,NaN,1984-11-18
1,1985,25,1106,77,1354,70,H,0,1984-10-29,NaN,NaN,1984-11-23
2,1985,25,1112,63,1223,56,H,0,1984-10-29,NaN,NaN,1984-11-23
3,1985,25,1165,70,1432,54,H,0,1984-10-29,NaN,NaN,1984-11-23
4,1985,25,1192,86,1447,74,H,0,1984-10-29,NaN,NaN,1984-11-23


In [47]:
total_count = len(
    tourney_results_odds_df[
        (tourney_results_odds_df["Season"] >= 2009)
        & (tourney_results_odds_df["Season"] < 2025)
    ]
)
odds_count = len(
    tourney_results_odds_df[
        (tourney_results_odds_df["Season"] >= 2009)
        & (tourney_results_odds_df["Season"] < 2025)
        & (tourney_results_odds_df["WOdds"].notna())
    ]
)
print(f"total_count: {total_count} \nodds_count: {odds_count}")

total_count: 998 
odds_count: 997


In [48]:
# Merge Team Names into compact_results_df
tourney_results_odds_df = tourney_results_odds_df.merge(
    teams_df[["TeamID", "TeamName"]], left_on="WTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "WTeamName"})
tourney_results_odds_df = tourney_results_odds_df.merge(
    teams_df[["TeamID", "TeamName"]], left_on="LTeamID", right_on="TeamID", how="left"
).rename(columns={"TeamName": "LTeamName"})
tourney_results_odds_df = tourney_results_odds_df.drop(
    columns=["TeamID_x", "TeamID_y"], errors="ignore"
)

In [49]:
tourney_results_odds_df[
    (tourney_results_odds_df["Season"] >= 2009)
    & (tourney_results_odds_df["Season"] < 2025)
    & (tourney_results_odds_df["WOdds"].isna())
]

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date,WTeamName,LTeamName
1628,2010,139,1326,75,1210,66,N,0,2009-11-02,NaN,NaN,2010-03-21,Ohio St,Georgia Tech


In [50]:
len(
    tourney_results_odds_df[
        tourney_results_odds_df.duplicated(
            subset=["Season", "DayNum", "WTeamID", "LTeamID"], keep=False
        )
    ]
)

0

In [51]:
total_count = len(
    regular_season_results_odds_df[
        (regular_season_results_odds_df["Season"] >= 2009)
        & (regular_season_results_odds_df["Season"] < 2025)
    ]
)
odds_count = len(
    regular_season_results_odds_df[
        (regular_season_results_odds_df["Season"] >= 2009)
        & (regular_season_results_odds_df["Season"] < 2025)
        & (regular_season_results_odds_df["WOdds"].notna())
    ]
)
print(f"total_count: {total_count} \nodds_count: {odds_count}")

total_count: 84416 
odds_count: 83207


In [52]:
len(
    regular_season_results_odds_df[
        regular_season_results_odds_df.duplicated(
            subset=["Season", "DayNum", "WTeamID", "LTeamID"], keep=False
        )
    ]
)

0

In [53]:
regular_season_results_odds_df.to_csv(
    r"data\betexplorer\MRegularSeasonCompactResultsOdds.csv", index=False
)
tourney_results_odds_df.to_csv(
    r"data\betexplorer\MNCAATourneyCompactResultsOdds.csv", index=False
)

### Add goto_conversion Probabilities

In [25]:
import pandas as pd
import numpy as np
import goto_conversion

# def apply_goto_conversion(df):
#     """
#     Applies the goto_conversion function to each row of a DataFrame with WOdds and LOdds.
#     Handles missing values by setting WProbability and LProbability to NaN if either WOdds or LOdds is missing.
#     Returns the DataFrame with added WProbability and LProbability columns.
#     """

#     def convert_row(row):
#         # Check if WOdds or LOdds is missing
#         if pd.isnull(row['WOdds']) or pd.isnull(row['LOdds']):
#             return pd.Series({'WProbability': np.nan, 'LProbability': np.nan})

#         try:
#             # Extract odds from the row
#             listOfOdds = [row['WOdds'], row['LOdds']]

#             # Compute probabilities using goto_conversion
#             probabilities = goto_conversion.goto_conversion(
#                 listOfOdds,
#                 total=1.0,
#                 multiplicativeIfUnprudentOdds=True,
#                 isAmericanOdds=False
#             )

#             # Assign probabilities to the respective teams
#             return pd.Series({'WProbability': probabilities[0], 'LProbability': probabilities[1]})

#         except Exception as e:
#             # Handle unexpected errors gracefully
#             print(f"Error processing row {row.name}: {e}")
#             return pd.Series({'WProbability': np.nan, 'LProbability': np.nan})

#     # Apply conversion function to each row
#     df[['WProbability', 'LProbability']] = df.apply(convert_row, axis=1)

#     return df


def apply_goto_conversion(df):
    """
    Applies the goto_conversion function to each row of a DataFrame with WOdds and LOdds.
    Handles missing or non-numeric values gracefully.
    Returns the DataFrame with added WProbability and LProbability columns.
    """

    def convert_row(row):
        try:
            # Convert WOdds and LOdds to floats (handles string values)
            WOdds = float(row["WOdds"]) if not pd.isnull(row["WOdds"]) else np.nan
            LOdds = float(row["LOdds"]) if not pd.isnull(row["LOdds"]) else np.nan

            # If odds are exactly 1.00, adjust them slightly to avoid errors
            if WOdds == 1.00:
                WOdds += 1e-6  # Tiny increment
            if LOdds == 1.00:
                LOdds += 1e-6  # Tiny increment

            # Check if either value is NaN (skip processing)
            if np.isnan(WOdds) or np.isnan(LOdds):
                return pd.Series({"WProbability": np.nan, "LProbability": np.nan})

            # Compute probabilities using goto_conversion
            listOfOdds = [WOdds, LOdds]
            probabilities = goto_conversion.goto_conversion(
                listOfOdds,
                total=1.0,
                multiplicativeIfUnprudentOdds=True,
                isAmericanOdds=False,
            )

            # Assign probabilities to the respective teams
            return pd.Series(
                {"WProbability": probabilities[0], "LProbability": probabilities[1]}
            )

        except Exception as e:
            # Handle unexpected errors gracefully
            print(f"Error processing row {row.name}: {e}")
            return pd.Series({"WProbability": np.nan, "LProbability": np.nan})

    # Convert entire columns to float (fix for entire dataset)
    df["WOdds"] = pd.to_numeric(df["WOdds"], errors="coerce")
    df["LOdds"] = pd.to_numeric(df["LOdds"], errors="coerce")

    # Apply conversion function to each row
    df[["WProbability", "LProbability"]] = df.apply(convert_row, axis=1)

    return df

In [32]:
regular_season_results_probs_df[114493:].head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date,WProbability,LProbability
114493,2011,33,1277,74,1132,39,H,0,2010-11-01,1.00,26.00,2010-12-04,0.999960,0.000040
114494,2011,33,1278,71,1165,66,H,0,2010-11-01,1.04,11.58,2010-12-04,0.953385,0.046615
114495,2011,33,1279,86,1379,81,H,0,2010-11-01,1.50,2.62,2010-12-04,0.646198,0.353802
114496,2011,33,1283,101,1146,61,H,0,2010-11-01,NaN,NaN,2010-12-04,NaN,NaN
114497,2011,33,1287,75,1293,65,H,0,2010-11-01,1.59,2.40,2010-12-04,0.608700,0.391300


In [30]:
regular_season_results_probs_df = apply_goto_conversion(regular_season_results_odds_df)
tourney_results_probs_df = apply_goto_conversion(tourney_results_odds_df)

In [33]:
regular_season_results_probs_df.to_csv(
    r"data\betexplorer\MRegularSeasonCompactResultsProbs.csv", index=False
)
tourney_results_probs_df.to_csv(
    r"data\betexplorer\MNCAATourneyCompactResultsProbs.csv", index=False
)

In [34]:
tourney_results_probs_df.head()

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,LOdds,match_date,WProbability,LProbability
0,1985,136,1116,63,1234,54,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
1,1985,136,1120,59,1345,58,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
2,1985,136,1207,68,1250,43,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
3,1985,136,1229,58,1425,55,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN
4,1985,136,1242,49,1325,38,N,0,1984-10-29,NaN,NaN,1985-03-14,NaN,NaN


In [12]:
from src.data_processing import extract_game_info, extract_seed_value

m_seed = pd.read_csv(r"data\kaggle\MNCAATourneySeeds.csv")
m_seed["SeedValue"] = m_seed["Seed"].apply(extract_seed_value)
tourney_round_lookup = pd.read_csv(r"data\tourney_round_lookup.csv")
tourney_results_probs_df = pd.read_csv(
    r"data\betexplorer\MNCAATourneyCompactResultsProbs.csv"
)


# Merge with seed_df to get Seed Values
def process_results(results_df, seed_df, tourney_round_lookup):
    results_df = results_df.merge(
        seed_df,
        left_on=["Season", "WTeamID"],
        right_on=["Season", "TeamID"],
        how="inner",
    )
    results_df = results_df.rename(
        columns={"SeedValue": "WSeedValue", "Seed": "WSeed"}
    ).drop(columns=["TeamID"])

    results_df = results_df.merge(
        seed_df,
        left_on=["Season", "LTeamID"],
        right_on=["Season", "TeamID"],
        how="inner",
    )
    results_df = results_df.rename(
        columns={"SeedValue": "LSeedValue", "Seed": "LSeed"}
    ).drop(columns=["TeamID"])

    # Ensure TeamA has the lower ID
    results_df["TeamA"] = results_df[["WTeamID", "LTeamID"]].min(axis=1)
    results_df["TeamB"] = results_df[["WTeamID", "LTeamID"]].max(axis=1)

    # Generate Matchup ID
    results_df["MatchupID"] = results_df.apply(
        lambda row: f"{row['Season']}_{row['TeamA']}_{row['TeamB']}", axis=1
    )

    # Compute Seed Difference
    results_df["SeedDiff"] = results_df["WSeedValue"].astype(int) - results_df[
        "LSeedValue"
    ].astype(int)
    # results_df['SeedDiff_Squared'] = results_df['SeedDiff'] ** 2

    # Define Outcome (1 if TeamA won, 0 otherwise)
    results_df["Outcome"] = (results_df["TeamA"] == results_df["WTeamID"]).astype(int)

    # # Create a flipped version from TeamB’s perspective
    # flipped_results = results_df.copy()
    # flipped_results['SeedDiff'] = -flipped_results['SeedDiff']
    # flipped_results['Outcome'] = 1 - flipped_results['Outcome']

    # # Combine original and flipped datasets
    # final_results = pd.concat([results_df, flipped_results], axis=0).reset_index(drop=True)

    # return final_results[['MatchupID', 'Season', 'SeedDiff', 'SeedDiff_Squared', 'Outcome']]

    # Ensure StrongSeed is alphabetically first, and WeakSeed is second
    results_df["StrongSeed"] = results_df[["WSeed", "LSeed"]].min(axis=1)
    results_df["WeakSeed"] = results_df[["WSeed", "LSeed"]].max(axis=1)

    results_df = results_df.merge(
        tourney_round_lookup,
        left_on=["StrongSeed", "WeakSeed"],
        right_on=["StrongSeed", "WeakSeed"],
        how="left",
    )

    return results_df


# Merge seed data for both men's and women's submissions
tourney_results_probs_df = process_results(
    tourney_results_probs_df, m_seed, tourney_round_lookup
)

In [4]:
tourney_results_probs_df[
    (
        tourney_results_probs_df["WProbability"]
        < tourney_results_probs_df["LProbability"]
    )
    & (tourney_results_probs_df["Round"] == 1)
]

,Season,DayNum,WTeamID,WScore,LTeamID,LScore,WLoc,NumOT,DayZero,WOdds,...,LSeedValue,TeamA,TeamB,MatchupID,SeedDiff,Outcome,StrongSeed,WeakSeed,Round,Slot
1591,2010,136,1293,66,1435,65,N,0,2009-11-02,2.22,...,4,1293,1435,2010_1293_1435,9,1,Z04,Z13,1.0,R1Z4
1594,2010,136,1325,97,1207,83,N,0,2009-11-02,9.02,...,3,1207,1325,2010_1207_1325,11,0,Y03,Y14,1.0,R1Y3
1595,2010,136,1330,51,1323,50,N,0,2009-11-02,2.16,...,6,1323,1330,2010_1323_1330,5,0,X06,X11,1.0,R1X6
1596,2010,136,1388,80,1350,71,N,0,2009-11-02,2.06,...,7,1350,1388,2010_1350_1388,3,0,X07,X10,1.0,R1X7
1599,2010,136,1448,81,1400,80,N,1,2009-11-02,2.86,...,8,1400,1448,2010_1400_1448,1,0,W08,W09,1.0,R1W8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2477,2024,137,1213,75,1388,66,N,0,2023-11-06,2.88,...,5,1213,1388,2024_1213_1388,7,1,X05,X12,1.0,R1X5
2479,2024,137,1241,72,1458,61,N,0,2023-11-06,2.65,...,5,1241,1458,2024_1241_1458,7,1,Z05,Z12,1.0,R1Z5
2481,2024,137,1321,77,1194,65,N,0,2023-11-06,2.57,...,8,1194,1321,2024_1194_1321,1,0,W08,W09,1.0,R1W8
2485,2024,137,1429,88,1395,72,N,0,2023-11-06,2.43,...,9,1395,1429,2024_1395_1429,-1,0,Y08,Y09,1.0,R1Y8


In [53]:
# tourney_results_probs_df = tourney_results_probs_df.drop(columns=['DayNum', 'DayZero'])
tourney_results_probs_df = tourney_results_probs_df.drop(columns=["WLoc"])

In [59]:
tourney_results_probs_df[
    (
        tourney_results_probs_df["WProbability"]
        > tourney_results_probs_df["LProbability"]
    )
    & (tourney_results_probs_df["SeedDiff"] > 0)
    & (tourney_results_probs_df["Round"] == 1)
]

,Season,WTeamID,WScore,LTeamID,LScore,NumOT,WOdds,LOdds,match_date,WProbability,...,LSeedValue,TeamA,TeamB,MatchupID,SeedDiff,Outcome,StrongSeed,WeakSeed,Round,Slot
1593,2010,1320,69,1424,66,0,1.88,1.92,2010-03-18,0.505695,...,8,1320,1424,2010_1320_1424,1,1,Y08,Y09,1.0,R1Y8
1657,2011,1211,86,1385,71,0,1.87,1.92,2011-03-17,0.507168,...,6,1211,1385,2011_1211_1385,5,1,Y06,Y11,1.0,R1Y6
1670,2011,1199,57,1401,50,0,1.86,1.90,2011-03-18,0.505853,...,7,1199,1401,2011_1199_1401,3,1,Z07,Z10,1.0,R1Z7
1743,2012,1301,79,1361,65,0,1.66,2.19,2012-03-16,0.575195,...,6,1301,1361,2012_1301_1361,5,1,X06,X11,1.0,R1X6
1808,2013,1235,76,1323,58,0,1.87,1.89,2013-03-22,0.502926,...,7,1235,1323,2013_1235_1323,3,1,Z07,Z10,1.0,R1Z7
1812,2013,1278,83,1417,63,0,1.55,2.40,2013-03-22,0.618069,...,6,1278,1417,2013_1278_1417,5,1,X06,X11,1.0,R1X6
1862,2014,1338,77,1160,48,0,1.33,3.37,2014-03-20,0.733764,...,8,1160,1338,2014_1160_1338,1,0,X08,X09,1.0,R1X8
1930,2015,1326,75,1433,72,1,1.55,2.47,2015-03-19,0.623367,...,7,1326,1433,2015_1326_1433,3,1,Z07,Z10,1.0,R1Z7
1988,2016,1139,71,1403,61,0,1.54,2.50,2016-03-17,0.627969,...,8,1139,1403,2016_1139_1403,1,1,X08,X09,1.0,R1X8
1989,2016,1163,74,1160,67,0,1.56,2.45,2016-03-17,0.619489,...,8,1160,1163,2016_1160_1163,1,0,Y08,Y09,1.0,R1Y8


In [13]:
tourney_results_probs_df = tourney_results_probs_df.drop(
    columns=[
        "DayNum",
        "WLoc",
        "DayZero",
        "TeamA",
        "TeamB",
        "WSeedValue",
        "SeedDiff",
        "Slot",
        "StrongSeed",
        "WeakSeed",
        "MatchupID",
    ]
)

In [17]:
tourney_results_probs_df[
    (tourney_results_probs_df["Round"] == 1)
    & (tourney_results_probs_df["LSeedValue"] == 1)
]

,Season,WTeamID,WScore,LTeamID,LScore,NumOT,WOdds,LOdds,match_date,WProbability,LProbability,WSeed,LSeed,LSeedValue,Outcome,Round
2150,2018,1420,74,1438,54,0,16.93,1.01,2018-03-16,0.014475,0.985525,Y16,Y01,1,1,1.0
